# 02 训练与评估

对应教程章节：[04-训练入门](../04-训练入门.md)、[05-看懂指标](../05-看懂指标.md)

**本 notebook 的终点**：跑通一次完整训练，读懂日志与指标，知道瓶颈在哪。

> **为什么训练用脚本、而不是内联 cell？**
> 一次 baseline 训练要 30~60 分钟。直接调 `model.train()` 会让内核一直占着内存、进度条刷爆输出。
> 这里改成 notebook **调用配套脚本**：脚本崩了不影响内核，长任务也能随时中断续训。
> 想学内联写法的话，第 7 节有对照示例。

## 0. 环境自检

In [ ]:
# ===== 环境自检与路径准备：每个 notebook 都先运行这一格 =====
import os, sys, pathlib, subprocess

# 0) 记下 notebook 启动时所在目录（后面用它找配套脚本），再切到作业根目录
START_DIR = pathlib.Path.cwd()

# 1) 缓存目录指到可写位置，避免 matplotlib / ultralytics 的权限报错
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplcache")
os.environ.setdefault("YOLO_CONFIG_DIR", "/tmp/ultralytics")

# 2) 定位作业根目录：向上查找含 data/ 或 datasets/ 或 codebase/ 的目录
def find_root(start=None):
    p = pathlib.Path(start or os.getcwd()).resolve()
    for cand in (p, *p.parents):
        if any((cand / d).is_dir() for d in ("data", "datasets", "codebase")):
            return cand
    return p

ROOT = find_root()
os.chdir(ROOT)
print("作业根目录:", ROOT)

# 3) 与配套脚本一致：优先使用作业自带的 ultralytics 源码
CB = ROOT / "codebase" / "ultralytics-YOLO26"
if CB.is_dir() and str(CB) not in sys.path:
    sys.path.insert(0, str(CB))

# 4) 依赖与 GPU 自检
missing = []
for mod, name in (("torch", "PyTorch"), ("ultralytics", "ultralytics"),
                  ("PIL", "Pillow"), ("matplotlib", "matplotlib")):
    try:
        __import__(mod)
    except ImportError:
        missing.append(name)
if missing:
    print("❌ 缺少依赖:", missing)
    print("   请在 ai-env 里执行：conda activate ai-env && pip install ipykernel matplotlib")
else:
    import torch, ultralytics
    print("✅ 依赖齐全")
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "（无 GPU，训练会非常慢）"
    print("CUDA 可用:", torch.cuda.is_available(), "|", gpu)
    print("ultralytics:", ultralytics.__version__, "->", ultralytics.__file__)


In [ ]:
# ===== 工具函数：找脚本 / 运行脚本 / 显示图片 =====
# 画图后端：在 Jupyter / IPython 里必须用 inline，否则 plt.show() 不会把图显示出来
# （纯脚本环境没有 IPython，就退回 Agg —— 不弹窗、只保存文件）
try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

def script(name):
    """定位配套脚本，兼容三种常见目录布局：

    1. <作业根>/code/train.py      —— 教程默认布局（把 code/ 拷进作业根）
    2. <作业根>/train.py           —— 把脚本直接放在作业根
    3. <notebook 同级的 code/>     —— 直接在教程文件夹里运行 notebook
    """
    cands = [ROOT / "code" / name, ROOT / name,
             START_DIR.parent / "code" / name, START_DIR / "code" / name]
    for c in cands:
        if c.exists():
            return str(c)
    raise FileNotFoundError(
        f"找不到 {name}。请把教程的 code/ 目录复制到作业根目录（与 data/ 同级），"
        f"或把脚本直接放在作业根目录。已查找：{[str(c) for c in cands]}")

def run(*args, check=True):
    """用当前内核的 Python 运行配套脚本，输出直接显示在下面。"""
    cmd = [sys.executable, *[str(a) for a in args]]
    print("$", " ".join(cmd), "\n")
    return subprocess.run(cmd, check=check)

def show(path, width=800, title=None):
    """在 notebook 里显示一张本地图片。

    图片文件直接用 IPython.display 显示（不经过 matplotlib，最稳）；
    不在 Jupyter 里时退回 PIL + matplotlib，至少不会报错。
    """
    path = pathlib.Path(path)
    if not path.exists():
        print("⚠️ 文件不存在:", path, "（先运行生成它的那一格）")
        return
    if title:
        print(title)
    try:
        from IPython.display import Image as _IPyImage, display
        display(_IPyImage(filename=str(path), width=width))
        return
    except ImportError:
        pass
    img = Image.open(path)
    w, h = img.size
    fig, ax = plt.subplots(figsize=(8, 8 * h / w))
    ax.imshow(img); ax.axis("off")
    if title:
        ax.set_title(title)
    plt.show()


## 1. 冒烟测试（别跳过）

用 2% 数据跑 1 个 epoch，几十秒就能暴露大部分问题（标签越界、路径写错、显存不够）。
跳过它，你可能在 20 分钟后才发现数据有问题。

In [ ]:
run(script("train.py"), "--name", "smoke", "--epochs", "1", "--fraction", "0.02",
    "--imgsz", "320", "--batch", "4", "--workers", "2", "--device", "0")

**这一轮的 mAP 没有参考价值**（1 轮 + 2% 数据），它只用来验证"链路通不通"。

✅ 成功的标志：`runs/smoke/weights/best.pt` 出现、`runs/smoke/results.csv` 有内容。

## 2. baseline 训练

baseline 就是"什么都不调"的基准，后面任何改动都跟它比：`imgsz=640`、`batch=8`、`epochs=200`（带早停）。

⏱ **本教程环境下约 36 分钟**（11 秒/轮，实际第 196 轮早停）。可以先去写文档。

In [ ]:
run(script("train.py"), "--name", "baseline_n_640", "--epochs", "200", "--imgsz", "640",
    "--batch", "8", "--workers", "2", "--device", "0")

### 训练时另开一个终端盯资源

```bash
watch -n 2 'free -h | head -2; nvidia-smi --query-gpu=utilization.gpu,memory.used,temperature.gpu,power.draw --format=csv,noheader'
```

三条经验线：内存 `used` 别超 85%（超了就 Ctrl+C 并降 `--workers`）；GPU 利用率长期低于 50% 说明瓶颈在数据读取
（用 `--cache`，不要加 workers）；温度长期高于 90℃ 要改善散热。

### 被打断了怎么办

```bash
python code/train.py --name baseline_n_640 --resume
```

续训只改资源类参数（batch/workers/device），**不要改 imgsz**——那等于换任务，脚本里的护栏会直接拦住你。

## 3. 读懂训练曲线

In [ ]:
show(ROOT / "runs/baseline_n_640/results.png", width=1400,
     title="训练曲线：上排是训练侧损失与 P/R，下排是验证侧损失与 mAP")

In [ ]:
import csv

src = ROOT / "runs/baseline_n_640/results.csv"
if not src.exists():
    print(f"⚠️ 还没找到 {src}")
    print("   请先运行上面那格训练（或者把 --name 换成你已经训练过的实验名）。")
else:
    rows = list(csv.DictReader(open(src)))
    print(f"共 {len(rows)} 轮")
    best = max(rows, key=lambda r: float(r["metrics/mAP50-95(B)"]))
    print(f"最佳轮次: 第 {best['epoch']} 轮  mAP50={float(best['metrics/mAP50(B)']):.4f}  "
          f"mAP50-95={float(best['metrics/mAP50-95(B)']):.4f}")
    print("\n epoch |  mAP50  | mAP50-95 | train_cls | val_cls")
    for ep in (1, 10, 25, 50, 100, int(best["epoch"]), len(rows)):
        r = rows[ep - 1]
        print(f"{int(r['epoch']):6d} | {float(r['metrics/mAP50(B)']):.4f} | "
              f"{float(r['metrics/mAP50-95(B)']):.4f}   | {float(r['train/cls_loss']):.3f}     | "
              f"{float(r['val/cls_loss']):.3f}")

**参考结果**：第 1 轮 mAP50 = 0.086 → 第 10 轮 0.868 → 第 146 轮最优 0.937 → 第 196 轮早停。

三个必须会的判断：

1. **前 10 轮 mAP50 就该到 0.8 以上**（迁移学习的正常表现）。10 轮后还贴着 0.1，先查数据和标注，别调参。
2. **训练损失降、验证损失升 = 过拟合**。本次第 146 轮后训练 cls_loss 继续降（0.478→0.414）、验证反而升（0.681→0.741），
   所以 `best.pt` 停在 146 轮。**提交要交 `best.pt`，不是 `last.pt`。**
3. **mAP50 高、mAP50-95 低 = 定位精度不足**。baseline 是 0.937 / 0.636，这个 0.3 的落差就是后面的调参方向。

In [ ]:
if "rows" in globals() and rows:
    eps = [int(r["epoch"]) for r in rows]
    m50 = [float(r["metrics/mAP50(B)"]) for r in rows]
    m95 = [float(r["metrics/mAP50-95(B)"]) for r in rows]

    plt.figure(figsize=(10, 4))
    plt.plot(eps, m50, label="mAP50")
    plt.plot(eps, m95, label="mAP50-95")
    plt.axvline(int(best["epoch"]), color="gray", ls="--", lw=1)
    plt.text(int(best["epoch"]) + 3, 0.2, f"best epoch={best['epoch']}", color="gray")
    plt.xlabel("epoch"); plt.ylabel("mAP"); plt.legend(); plt.grid(alpha=0.3)
    plt.title("baseline_n_640：mAP 随 epoch 变化"); plt.show()
else:
    print("⚠️ 还没有逐轮数据，先运行上一格。")

## 4. 评估：拿到 mAP50 / mAP75 / mAP50-95

⚠️ **训练日志里只有 mAP50 与 mAP50-95，没有 mAP75**。作业要求报 mAP75 时，必须用 `eval.py` 单独评估一次
（它读的是框架内部的 `metrics.box.map75`）。

先用 val（可以反复用），最后才用 test（只用一次）。

In [ ]:
run(script("eval.py"), "--weights", "runs/baseline_n_640/weights/best.pt",
    "--split", "val", "--imgsz", "640", "--batch", "8", "--device", "0")

In [ ]:
# 所有实验的指标会自动汇总到 runs/summary.csv —— 这就是作业表格的数据来源
print((ROOT / "runs/summary.csv").read_text())

## 5. 看 PR 曲线与混淆矩阵

In [ ]:
ev = ROOT / "runs/val/baseline_n_640_val_640"
show(ev / "BoxPR_curve.png", width=800, title="PR 曲线（越靠右上越好）")
show(ev / "confusion_matrix_normalized.png", width=700, title="混淆矩阵（归一化）")

**怎么读混淆矩阵**（本次 test 集的真实数字）：

| | 真实是 robot | 真实是背景 |
|---|---|---|
| 预测 robot | 963（正确检出） | 76（误检） |
| 预测背景 | **73（漏检）** | — |

1036 个目标里漏检 73 个（7%）、多报 76 个，误差分散在多种场景，属正常水平。

- 漏检多 → 查是否漏标、目标是否太小、是否需要提高分辨率
- 误检多 → 查是否有大量相似背景，或标注把背景框了进去

> 混淆矩阵的判定阈值是 conf=0.25、IoU=0.45；换阈值数字会变，比较时要说清口径。

## 6. 找出到底哪张图错了

In [ ]:
run(script("analyze_errors.py"), "--weights", "runs/baseline_n_640/weights/best.pt",
    "--split", "test", "--imgsz", "640")

In [ ]:
import glob
for p in sorted(glob.glob(str(ROOT / "runs/error_analysis/test_*.jpg")))[:2]:
    show(p, width=900, title=pathlib.Path(p).name)

**本次的真实发现**：一张拆解状态的机器人零件堆，模型其实检测到了（置信度 0.406），但预测框把上方的背包和器材
也框了进去，与标注框 IoU 只有 0.529——在 mAP50 上算命中，在高 IoU 上丢分。这就是"框包容过多背景"的典型失效模式，
改进方向是补充遮挡与杂物场景的数据。

## 7. （选学）内联写法对照

想在 notebook 里直接训练也可以，但要接受两个代价：内核内存持续占用、进度条刷屏。

```python
from ultralytics import YOLO

model = YOLO(str(ROOT / "codebase/ultralytics-YOLO26/yolo26n.pt"))
model.train(
    data=str(ROOT / "datasets/rm/data.yaml"),
    epochs=50, imgsz=640, batch=8, workers=2, device=0,
    project=str(ROOT / "runs"), name="inline_demo", exist_ok=True,
)
```

参数含义与 `code/train.py` 完全一致。**长训练仍建议用脚本。**

## 检查点

- [ ] 冒烟测试通过，`runs/smoke/weights/best.pt` 存在
- [ ] baseline 训练跑完，`results.png` 能看懂
- [ ] 能解释 `box_loss / cls_loss / dfl_loss` 分别是什么
- [ ] 知道为什么交 `best.pt` 而不是 `last.pt`
- [ ] 用 `eval.py` 拿到了 mAP50 / mAP75 / mAP50-95
- [ ] 看过 PR 曲线与混淆矩阵，知道漏检/误检数量

继续 **03-调参与交付**。